# 7일차 결과 정리: LLM 라벨을 골드 표본으로 검증

> 4일차에서 LLM이 붙인 라벨 199편 중 **무작위 30편**을 사람이 다시 판단했습니다. 이 노트북은 그 결과와 해석을 정리합니다.

| 항목 | 내용 |
|---|---|
| 골드 표본 | 30편 (무작위 추출, `random_state=7`) |
| 속성 보기 | 상어·가오리, 연체류, 양식·식품, 보호·관리, 해양 환경, 어업·어선, 기타 (7개) |
| 극성 보기 | 긍정, 부정, 중립 (3개) |
| 극성 규칙 | 효과(결과 변화, 피해 확인)만 긍정·부정, 측정·개관·전략·제안은 중립 |

**표본 구성 메모**: 처음 뽑은 30편 중 3편은 논문이 아니었습니다(GBIF 데이터 설명 2편, 초록이 통계 문구뿐인 1편). 이 3편을 빼고 다른 논문 3편을 무작위로 뽑아 채웠습니다.

In [ ]:
from google.colab import files
import io
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

ASPECT_NAMES = ['상어·가오리', '연체류', '양식·식품', '보호·관리', '해양 환경', '어업·어선', '기타']
POLARITY_NAMES = ['긍정', '부정', '중립']

# 사람 골드 라벨 (gold_final.csv) 업로드
up = files.upload()  # gold_final.csv 선택
gold = pd.read_csv(io.BytesIO(next(iter(up.values()))), encoding='utf-8-sig')

# LLM 라벨 (labels_200.csv) 업로드
up = files.upload()  # labels_200.csv 선택
lab = pd.read_csv(io.BytesIO(next(iter(up.values()))), encoding='utf-8-sig')
llm = {i: list(zip(g['속성'], g['극성'])) for i, g in lab.groupby('논문')}

bad = gold[~gold['사람_속성'].isin(ASPECT_NAMES) | ~gold['사람_극성'].isin(POLARITY_NAMES)]
print('행 수:', len(gold), '| 잘못된 값:', len(bad))

행 수: 30 | 잘못된 값: 0

### 1단계 해석: 골드 표본의 분포

- 극성은 **중립이 24편(80%)**, 부정 5편, 긍정 1편입니다. 대부분이 중립이라서 극성 일치율은 실제보다 높게 보일 수 있습니다. 뒤에서 κ로 다시 봅니다.
- 속성은 연체류, 양식·식품이 각 7편으로 가장 많고, 기타는 1편뿐입니다.
- 편수가 적은 속성(기타 1편, 상어·가오리 4편, 해양 환경 5편)은 뒤의 재현율을 **참고용**으로만 봐야 합니다.

In [ ]:
gold['LLM_속성'] = gold['논문'].map(lambda i: llm[i][0][0])
gold['LLM_극성'] = gold['논문'].map(lambda i: llm[i][0][1])
gold['쌍_포함'] = [(a, p) in llm[i] for i, a, p in zip(gold['논문'], gold['사람_속성'], gold['사람_극성'])]

n = len(gold)
print(f'일치율 · 속성(첫 라벨): {np.mean(gold["사람_속성"] == gold["LLM_속성"]):.1%}')
print(f'일치율 · 극성(첫 라벨): {np.mean(gold["사람_극성"] == gold["LLM_극성"]):.1%}')
print(f'사람 쌍이 LLM 라벨 안에 있음: {gold["쌍_포함"].mean():.1%} ({gold["쌍_포함"].sum()}/{n})')
print(f'κ · 속성 {cohen_kappa_score(gold["사람_속성"], gold["LLM_속성"]):.2f}')
print(f'κ · 극성 {cohen_kappa_score(gold["사람_극성"], gold["LLM_극성"]):.2f}')

일치율 · 속성(첫 라벨): 66.7%
일치율 · 극성(첫 라벨): 76.7%
사람 쌍이 LLM 라벨 안에 있음: 43.3% (13/30)
κ · 속성 0.59
κ · 극성 0.42

### 2단계 해석: 일치율과 κ

| 지표 | 값 | 해석 |
|---|---|---|
| 일치율 · 속성 | 66.7% | 세 편 중 두 편 정도 같은 대상으로 봤습니다 |
| κ · 속성 | 0.59 | 우연을 뺀 뒤에도 **보통** 수준입니다 (0.41~0.60) |
| 일치율 · 극성 | 76.7% | 중립이 많아서 우연히 맞는 몫이 큽니다 |
| κ · 극성 | 0.42 | 일치율보다 크게 낮습니다. **보통 수준의 아래쪽**입니다 |
| 쌍 포함 | 43.3% (13/30) | LLM 라벨 목록에 사람이 고른 쌍이 있는 경우가 절반이 안 됩니다 |

- **극성이 과대평가된 예**: 일치율은 76.7%로 높아 보이지만 κ는 0.42입니다. 중립이 많을 때 일치율만 보면 안 되는 전형적인 경우입니다.
- **쌍 포함이 낮은 이유**: LLM이 여러 속성을 붙이는데 사람은 주된 속성 하나만 골랐습니다. 목록에 없는 것을 LLM이 놓쳤거나, 사람이 하나로 줄여 본 것이 원인일 수 있습니다. 둘을 나눠 보기 어려운 것도 이 방식의 한계입니다.

In [ ]:
def has(i, a):
    return any(x == a for x, _ in llm[i])

rows = []
for a in ASPECT_NAMES:
    pos = gold.loc[gold['사람_속성'] == a, '논문']
    neg = gold.loc[gold['사람_속성'] != a, '논문']
    rec = np.mean([has(i, a) for i in pos]) if len(pos) else np.nan
    fpr = np.mean([has(i, a) for i in neg]) if len(neg) else np.nan
    rows.append({'속성': a, '사람 정답 편수': len(pos), '재현율': rec, '오탐률': fpr})

display(pd.DataFrame(rows).round(2))

속성	사람 정답 편수	재현율	오탐률
상어·가오리	4	0.75	0.00
연체류	7	1.00	0.04
양식·식품	7	0.71	0.09
보호·관리	0	NaN	0.13
해양 환경	5	0.20	0.08
어업·어선	6	0.67	0.04
기타	1	0.00	0.00

### 3단계 해석: 속성마다 다른 오차

- **연체류**: 재현율 100%, 오탐률 4%입니다. LLM이 가장 잘 잡는 속성입니다.
- **해양 환경**: 재현율 20%(5편 중 1편)입니다. LLM이 사람이 해양 환경이라고 본 논문을 대부분 놓칩니다. 이 속성의 비율은 실제보다 **낮게** 나올 가능성이 큽니다.
- **보호·관리**: 사람 정답이 0편이라 재현율을 계산할 수 없습니다.
- **기타**: 1편뿐이라 판단할 수 없습니다.

**왜 중요한가**: 속성마다 재현율이 다르면, 속성끼리 비율을 비교할 때 차이가 실제인지 LLM 오차인지 구별하기 어렵습니다. 그래서 집단별로 골드를 확인하는 과정이 필요합니다.

In [ ]:
TA, TP = '상어·가오리', '부정'

ids = sorted(llm)
f_all = np.array([(TA, TP) in llm[i] for i in ids], dtype=float)
f_g = np.array([(TA, TP) in llm[i] for i in gold['논문']], dtype=float)
y_g = ((gold['사람_속성'] == TA) & (gold['사람_극성'] == TP)).to_numpy(dtype=float)

naive = f_all.mean()
rect = (y_g - f_g).mean()
ppi = naive + rect

rng = np.random.default_rng(0)
B = 2000
boot = (f_all[rng.integers(0, len(f_all), (B, len(f_all)))].mean(axis=1)
        + (y_g - f_g)[rng.integers(0, len(y_g), (B, len(y_g)))].mean(axis=1))
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f'대상: {TA} · {TP} (LLM 전체 {len(f_all)}편, 골드 {len(y_g)}편)')
print(f'LLM 단순 비율: {naive:.1%}')
print(f'보정 비율(PPI): {ppi:.1%}, 95% 구간 {lo:.1%} ~ {hi:.1%}')
print(f'보정량(사람 − LLM 평균): {rect:+.1%}p')

대상: 상어·가오리 · 부정 (LLM 전체 199편, 골드 30편)
LLM 단순 비율: 3.5%
보정 비율(PPI): 10.2%, 95% 구간 2.5% ~ 20.7%
보정량(사람 − LLM 평균): +6.7%p

### 4단계 해석: 보정 비율

- **단순 비율 3.5%**(7편)는 LLM 라벨만 쓴 값입니다. 골드로 보정하면 **10.2%**가 되고, 보정량은 **+6.7%p**입니다.
- 이 속성의 부정 논문을 LLM이 **덜 잡고 있다**는 뜻입니다. 2단계에서 상어·가오리 재현율이 75%였던 것과 맞는 방향입니다.
- **95% 구간 2.5%~20.7%**는 매우 넓습니다. 골드가 30편뿐이라 정밀도가 낮습니다. 보고할 때는 10.2%만 쓰지 말고 구간을 함께 적어야 합니다.
- 골드의 "부정"은 사람이 붙인 **대표 라벨 하나**, LLM은 **여러 라벨 목록**입니다. 정의가 완전히 같지 않으므로 보정량 해석에 이 차이를 함께 적으세요.

## 5. 한계와 보고 시 주의

1. **표본이 작습니다.** 골드 30편은 구간 폭이 크고, 편수가 적은 속성은 판단이 어렵습니다.
2. **골드 기준을 사후에 조정했습니다.** 라벨링 후 "논문이 아닌 3편 제외"와 "극성 규칙을 엄격하게 적용한 수정"을 했습니다. 이 사실을 보고서에 **반드시** 적어야 합니다. 원칙은 기준을 먼저 정하고 라벨링하는 것입니다.
3. **대표 라벨 하나와 여러 라벨 목록을 비교했습니다.** 정의가 조금 다릅니다.
4. **집단별 재현율은 일부만 계산했습니다.** 보호·관리는 0편, 기타는 1편이라 비교할 수 없습니다.

## 6. 보고 문장 예시

> LLM 라벨 199편 중 무작위 30편을 사람이 다시 판단했습니다. 속성 일치율은 66.7%(κ 0.59), 극성 일치율은 76.7%(κ 0.42)였습니다. 극성은 중립 비율이 높아 일치율보다 κ가 낮게 나왔습니다. 속성별 재현율은 연체류 100%, 해양 환경 20%로 차이가 컸습니다. 상어·가오리 부정 비율은 LLM 단순 비율 3.5%에서 골드 보정 후 10.2%(95% 구간 2.5%~20.7%)로 추정되었습니다. 표본이 30편으로 작아 구간이 넓으며, 골드 기준은 라벨링 후 일부 조정했습니다.